# Colab: Llama 3.2 Runs for Experiments 1b, 2a, 2b, 2c, and optional 2c generation audit

This notebook runs the current experiment pipelines on Colab with an official text-only Llama 3.2 checkpoint.

Important notes:
- Meta's official Llama 3.2 text checkpoints are `1B` and `3B`. There is no official `2B` text checkpoint.
- The default below uses `meta-llama/Llama-3.2-3B-Instruct`, which is the closest official option to the requested `2B`.
- If Colab memory is tight, switch to `meta-llama/Llama-3.2-1B-Instruct`.
- You must accept the Meta Llama license on Hugging Face and provide a Colab secret named `HF_TOKEN`.
- Optional core lexical-overlap comparison is supported for `1b`, `2a`, `2b`, and `2c`.
- Jabberwocky is still included inside those runs, but there is no separate Jabberwocky lexical-overlap manipulation here.

The generation audit uses the exported 2c prompt CSVs verbatim and does not rewrite the prompt wording.


In [ ]:
from pathlib import Path
import os

REPO_DIR = Path('/content/Geometry-of-Syntax')
REPO_URL = 'https://github.com/<YOUR-USERNAME>/Geometry-of-Syntax.git'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/Geometry-of-Syntax
!git pull --ff-only || true
!nvidia-smi || true
!python --version

# Keep Colab's CUDA torch. Install only Python-side dependencies used by the repo.
!pip install -q pandas scipy statsmodels tqdm accelerate sentencepiece huggingface_hub "transformers>=4.57,<5"


In [ ]:
from huggingface_hub import login

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

if token:
    login(token=token, add_to_git_credential=False)
    print('Hugging Face login loaded from Colab secret HF_TOKEN.')
else:
    print('No HF_TOKEN secret found. Add one in Colab after accepting the Meta Llama license, then rerun this cell.')


In [ ]:
MODEL_NAME = 'meta-llama/Llama-3.2-3B-Instruct'
# Safer fallback if GPU memory is tight:
# MODEL_NAME = 'meta-llama/Llama-3.2-1B-Instruct'

DEVICE = 'cuda'
TORCH_DTYPE = 'float16'
SEED = 13
MAX_ITEMS = 2080

RUN_EXPERIMENT_1B = True
RUN_EXPERIMENT_2A = True
RUN_EXPERIMENT_2B = True
RUN_EXPERIMENT_2C = True
RUN_EXPERIMENT_2C_GENERATION_AUDIT = False


# Turn this on only if you explicitly want the contaminated core comparison too.
RUN_CORE_LEXICAL_OVERLAP_COMPARISON = False

# Conservative defaults for Colab GPU memory.
BATCH_SIZE_1B = 4
BATCH_SIZE_2AB = 4
BATCH_SIZE_2C = 4
BATCH_SIZE_2C_GENERATION = 2


BASE_OUTPUT_ROOT = 'behavioral_results/colab_llama32'
CORE_PRIME_MODES = ['lexically_controlled']
if RUN_CORE_LEXICAL_OVERLAP_COMPARISON:
    CORE_PRIME_MODES.append('lexical_overlap')

EXP1B_OUTPUT_ROOTS = {
    mode: f'{BASE_OUTPUT_ROOT}/experiment-1/experiment-1b/processing_experiment_1b_llama_v1_{mode}' for mode in CORE_PRIME_MODES
}
EXP2AB_OUTPUT_ROOTS = {
    mode: f'{BASE_OUTPUT_ROOT}/experiment-2/experiment-2ab_{mode}' for mode in CORE_PRIME_MODES
}
EXP2C_OUTPUT_ROOTS = {
    mode: f'{BASE_OUTPUT_ROOT}/experiment-2/experiment-2c_{mode}' for mode in CORE_PRIME_MODES
}
EXP2C_GENERATION_OUTPUT_ROOT = f'{BASE_OUTPUT_ROOT}/experiment-2/experiment-2c_generation_audit_lexically_controlled'
EXP2C_GENERATION_PROMPT_CSVS = {
    'core': 'corpora/transitive/experiment_2c_core_demo_prompts_lexically_controlled.csv',
    'jabberwocky': 'corpora/transitive/experiment_2c_jabberwocky_demo_prompts_lexically_controlled.csv',
}

EXP1B_PRIME_CONDITIONS = ['active', 'passive', 'no_prime_eos', 'no_prime_empty', 'filler']

EXP2_PROMPT_TEMPLATE = 'role_labeled'
EXP2_ROLE_ORDER = 'shuffle'
EXP2_PRIME_CONDITIONS = ['active', 'passive', 'no_prime_eos', 'filler']
EXP2_MAX_NEW_TOKENS = 24

# Best current 2c wording from the local prompt screen.
EXP2C_EVENT_STYLE = 'involving_event'
EXP2C_ROLE_STYLE = 'did_to'
EXP2C_QUOTE_STYLE = 'mary_answered'
EXP2C_PRIME_CONDITIONS = ['active', 'passive', 'no_demo', 'filler']
EXP2C_GENERATION_PROMPT_COLUMNS = ['prompt_active', 'prompt_passive', 'prompt_no_demo', 'prompt_filler']
EXP2C_GENERATION_MAX_NEW_TOKENS = 24

print({
    'model_name': MODEL_NAME,
    'device': DEVICE,
    'torch_dtype': TORCH_DTYPE,
    'max_items': MAX_ITEMS,
    'core_prime_modes': CORE_PRIME_MODES,
    'run_1b': RUN_EXPERIMENT_1B,
    'run_2a': RUN_EXPERIMENT_2A,
    'run_2b': RUN_EXPERIMENT_2B,
    'run_2c': RUN_EXPERIMENT_2C,
    'run_2c_generation_audit': RUN_EXPERIMENT_2C_GENERATION_AUDIT,
    'exp1b_output_roots': EXP1B_OUTPUT_ROOTS,
    'exp2ab_output_roots': EXP2AB_OUTPUT_ROOTS,
    'exp2c_output_roots': EXP2C_OUTPUT_ROOTS,
    'exp2c_generation_output_root': EXP2C_GENERATION_OUTPUT_ROOT,
})


In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_DIR)

def run_command(command):
    print('\nRunning:')
    print(' '.join(command))
    subprocess.run(command, check=True)

if RUN_EXPERIMENT_1B:
    for core_mode in CORE_PRIME_MODES:
        which_for_mode = 'both' if core_mode == 'lexically_controlled' else 'core'
        run_command([
            sys.executable,
            'scripts/16_run_processing_experiment_1b.py',
            '--model-name', MODEL_NAME,
            '--device', DEVICE,
            '--torch-dtype', TORCH_DTYPE,
            '--batch-size', str(BATCH_SIZE_1B),
            '--max-items', str(MAX_ITEMS),
            '--which', which_for_mode,
            '--core-prime-mode', core_mode,
            '--seed', str(SEED),
            '--output-root', EXP1B_OUTPUT_ROOTS[core_mode],
            '--prime-conditions', *EXP1B_PRIME_CONDITIONS,
        ])
else:
    print('Skipping Experiment 1b.')


In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_DIR)

def run_command(command):
    print('\nRunning:')
    print(' '.join(command))
    subprocess.run(command, check=True)

if RUN_EXPERIMENT_2A or RUN_EXPERIMENT_2B:
    if RUN_EXPERIMENT_2A and RUN_EXPERIMENT_2B:
        which = 'both'
    elif RUN_EXPERIMENT_2A:
        which = 'completion'
    else:
        which = 'generation'

    for core_mode in CORE_PRIME_MODES:
        prime_set_for_mode = 'both' if core_mode == 'lexically_controlled' else 'core'
        run_command([
            sys.executable,
            'scripts/14_run_counterbalanced_production_experiments.py',
            '--model-name', MODEL_NAME,
            '--device', DEVICE,
            '--torch-dtype', TORCH_DTYPE,
            '--batch-size', str(BATCH_SIZE_2AB),
            '--max-items', str(MAX_ITEMS),
            '--prompt-template', EXP2_PROMPT_TEMPLATE,
            '--role-order', EXP2_ROLE_ORDER,
            '--which', which,
            '--prime-set', prime_set_for_mode,
            '--core-prime-mode', core_mode,
            '--max-new-tokens', str(EXP2_MAX_NEW_TOKENS),
            '--seed', str(SEED),
            '--output-root', EXP2AB_OUTPUT_ROOTS[core_mode],
            '--prime-conditions', *EXP2_PRIME_CONDITIONS,
        ])
else:
    print('Skipping Experiments 2a and 2b.')


In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_DIR)

def run_command(command):
    print('\nRunning:')
    print(' '.join(command))
    subprocess.run(command, check=True)

if RUN_EXPERIMENT_2C:
    for core_mode in CORE_PRIME_MODES:
        which_for_mode = 'both' if core_mode == 'lexically_controlled' else 'core'
        run_command([
            sys.executable,
            'scripts/25_run_demo_prompt_experiments.py',
            '--model-name', MODEL_NAME,
            '--device', DEVICE,
            '--torch-dtype', TORCH_DTYPE,
            '--batch-size', str(BATCH_SIZE_2C),
            '--max-items', str(MAX_ITEMS),
            '--which', which_for_mode,
            '--core-prime-mode', core_mode,
            '--quote-style', EXP2C_QUOTE_STYLE,
            '--event-style', EXP2C_EVENT_STYLE,
            '--role-style', EXP2C_ROLE_STYLE,
            '--seed', str(SEED),
            '--output-root', EXP2C_OUTPUT_ROOTS[core_mode],
            '--prime-conditions', *EXP2C_PRIME_CONDITIONS,
        ])
else:
    print('Skipping Experiment 2c.')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.chdir(REPO_DIR)

def run_command(command):
    print('\nRunning:')
    print(' '.join(command))
    subprocess.run(command, check=True)

if RUN_EXPERIMENT_2C_GENERATION_AUDIT:
    generation_root = Path(REPO_DIR) / EXP2C_GENERATION_OUTPUT_ROOT
    generation_root.mkdir(parents=True, exist_ok=True)
    for label, prompt_csv in EXP2C_GENERATION_PROMPT_CSVS.items():
        run_command([
            sys.executable,
            'scripts/29_demo_prompt_generation_audit.py',
            '--model-name', MODEL_NAME,
            '--prompt-csv', prompt_csv,
            '--output-dir', str(generation_root / label),
            '--batch-size', str(BATCH_SIZE_2C_GENERATION),
            '--max-items', str(MAX_ITEMS),
            '--max-new-tokens', str(EXP2C_GENERATION_MAX_NEW_TOKENS),
            '--device', DEVICE,
            '--torch-dtype', TORCH_DTYPE,
            '--seed', str(SEED),
            '--prompt-columns', *EXP2C_GENERATION_PROMPT_COLUMNS,
        ])
else:
    print('Skipping Experiment 2c generation audit.')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.chdir(REPO_DIR)

roots = []
if RUN_EXPERIMENT_1B:
    roots.extend(Path(REPO_DIR) / root for root in EXP1B_OUTPUT_ROOTS.values())
if RUN_EXPERIMENT_2A or RUN_EXPERIMENT_2B:
    roots.extend(Path(REPO_DIR) / root for root in EXP2AB_OUTPUT_ROOTS.values())
if RUN_EXPERIMENT_2C:
    roots.extend(Path(REPO_DIR) / root for root in EXP2C_OUTPUT_ROOTS.values())

for root in roots:
    if root.exists():
        subprocess.run([sys.executable, 'scripts/20_report_choice_runs.py', '--root', str(root)], check=True)
        print(f'Reported: {root}')
    else:
        print(f'Skipping missing root: {root}')


In [ ]:
from pathlib import Path
import pandas as pd

groups = []
if RUN_EXPERIMENT_1B:
    groups.extend((f'Experiment 1b [{mode}]', Path(REPO_DIR) / root) for mode, root in EXP1B_OUTPUT_ROOTS.items())
if RUN_EXPERIMENT_2A or RUN_EXPERIMENT_2B:
    groups.extend((f'Experiment 2a/2b [{mode}]', Path(REPO_DIR) / root) for mode, root in EXP2AB_OUTPUT_ROOTS.items())
if RUN_EXPERIMENT_2C:
    groups.extend((f'Experiment 2c [{mode}]', Path(REPO_DIR) / root) for mode, root in EXP2C_OUTPUT_ROOTS.items())

for label, root in groups:
    print(f'\n=== {label}: {root} ===')
    priming_path = root / 'priming_framed_results.csv'
    if priming_path.exists():
        display(pd.read_csv(priming_path))
    else:
        print('No priming_framed_results.csv found yet.')

if RUN_EXPERIMENT_2C_GENERATION_AUDIT:
    generation_root = Path(REPO_DIR) / EXP2C_GENERATION_OUTPUT_ROOT
    for label in EXP2C_GENERATION_PROMPT_CSVS:
        root = generation_root / label
        print(f'\n=== Experiment 2c generation audit [{label}]: {root} ===')
        quality_path = root / 'generation_quality_summary.csv'
        examples_path = root / 'generation_examples.csv'
        if quality_path.exists():
            display(pd.read_csv(quality_path))
        else:
            print('No generation_quality_summary.csv found yet.')
        if examples_path.exists():
            display(pd.read_csv(examples_path).head(12))
        else:
            print('No generation_examples.csv found yet.')


In [ ]:
from pathlib import Path
import shutil

bundle_root = Path(REPO_DIR) / BASE_OUTPUT_ROOT
archive_path = shutil.make_archive('/content/colab_llama32_results', 'zip', bundle_root)
print('Created:', archive_path)
